# Telltale: train on Kaggle (M4 / M6)

Settings: **Accelerator = GPU T4 x2 or P100**, **Internet = on**. Add a Kaggle secret named `HF_TOKEN`
(Hugging Face token with write access). Training spends no HF credits; it only uses Kaggle's free GPU.

Before this: run `telltale data extract`, `telltale mirrors v0` (and `polish`) on any machine, then
`telltale data push <you>/telltale-data` so this notebook can pull the data.

In [ ]:
REPO_URL = "https://github.com/sivaratrisrinivas/tell-tale"   # or attach the repo zip as a Kaggle dataset
DATA_REPO = "Srini410/telltale-data"     # private HF dataset pushed by `telltale data push`
MODEL_REPO = "Srini410/telltale"         # private HF model repo for checkpoints + ONNX
ROUND = 0                                # 0 = v0; 1..3 = hard-negative rounds (mine + mirror first, then rerun)
BASE_MODEL = "answerdotai/ModernBERT-base"  # "jhu-clsp/mmBERT-small" if multilingual matters

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
if not os.path.exists("telltale"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "telltale"], check=True)
%cd telltale
!pip install -q -e ".[train]" 2>&1 | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Pull the pipeline artifacts (human items, mirrors, hard-negative ids, polished set) into runs/
from huggingface_hub import snapshot_download
import shutil, pathlib
d = snapshot_download(DATA_REPO, repo_type="dataset", allow_patterns=["raw/*", "raw/**/*"])
shutil.copytree(pathlib.Path(d) / "raw", "runs", dirs_exist_ok=True)
!find runs -maxdepth 2 -type f | head -50

In [ ]:
# Build the labeled table. From round 1 on, mined pool items + their mirrors join train.
flag = "--with-hard-negatives" if ROUND > 0 else ""
!telltale data build {flag}

In [ ]:
# Train from the BASE model every round (spec M6 step 5). fp16 on T4; evaluates every half epoch,
# keeps the best validation AUROC, then fits temperature scaling + a 1%-FPR threshold on validation.
out = f"runs/model/round{ROUND}"
!telltale train --base-model {BASE_MODEL} --out {out}

In [ ]:
# Evaluate T1-T7 (T5 needs runs/mirrors/polished.jsonl, T6 needs a hand-checked runs/data/t6_candidates.csv)
!telltale eval --model runs/model/round{ROUND}/best --out runs/reports/round{ROUND}.md --title "Telltale round {ROUND}"


In [ ]:
# Mine the next round's hard negatives from the 100K pool (scored with this model)
!telltale mine --model runs/model/round{ROUND}/best --round {ROUND + 1} --k 3000
print(open(f"runs/hard_neg/round{ROUND + 1}_analysis.md").read())
# Next: on a machine with HF_TOKEN run `telltale mirrors hardneg --round {ROUND+1}`, push data, set ROUND+=1, rerun.

In [ ]:
# Export ONNX int8 + latency check, then push checkpoint + ONNX to the Hub (private)
!telltale export --model runs/model/round{ROUND}/best --out runs/onnx
!telltale eval --model runs/onnx --out runs/reports/round{ROUND}_onnx.md --latency-onnx runs/onnx --title "Telltale round {ROUND} (ONNX int8)"
!telltale push-model {MODEL_REPO} --onnx-dir runs/onnx --torch-dir runs/model/round{ROUND}/best --card MODEL_CARD.md

In [ ]:
# Keep the reports and hard-negative ids: push them back to the data repo
from huggingface_hub import HfApi
api = HfApi()
for sub in ["reports", "hard_neg"]:
    if pathlib.Path(f"runs/{sub}").exists():
        api.upload_folder(repo_id=DATA_REPO, repo_type="dataset", folder_path=f"runs/{sub}", path_in_repo=f"raw/{sub}")